# README

## Issues

- BOP Toolkit (needed for generating metadata and final dataset processing) is not yet fully working for the new H3 data format which we use. See [github issue](https://github.com/thodan/bop_toolkit/issues/137)
- Inspection of symmetry annotation visualizations looks not so great... See [google drive folder](https://drive.google.com/file/d/1WFxA6jOA97R-ScvhSgTzL2jQlxGU7fxY/view?usp=drive_link)
    - see `convert_symmetries` function
- Meshlab remeshing scripts don't work with any new versions of pymeshlab, conversion of old script attempted but not working as expected. See `remesh_for_eval` function

## Documentation of Format

The IPD native format and BOP H3 format uses similar terminology in different ways. Here are some of the differences.

- IPD parts are considered BOP objects
    - See generated `ipd_part_to_bop_obj_id.json` for mapping
- IPD datasets are considered BOP scenes
    - See generated `ipd_dataset_to_bop_scene.json` for mapping
- IPD objects (referenced by part and instance) are considered BOP ground truth instances.
    - See generated `test/[bop_scene/ipd_obj_to_bop_gt_id.json` for mapping

Conversion details:
- IPD cameras are considered different BOP `split_type`s
- IPD lighting conditions are specified by different `test_targets`

```
/ipd
######## BASE ZIP
├─ camera_photoneo.json
├─ camera_basler_hr.json
├─ camera_basler_lr.json
├─ camera_flir_polar.json
├─ ipd_part_to_bop_obj_id.json
	- mapping from part name to BOP OBJ_ID
├─ ipd_dataset_to_bop_scene.json
	- mapping from dataset_id / background to BOP SCENE_ID
├─ test_targets_bop19_[all, room, day, spot].json
	- instances for each object in each scene, in each dataset, for different subsets of lighting conditions
########

######## MODELS ZIP
├─ models
│  ├─ models_info.json
│  ├─ obj_OBJ_ID.ply
├─ models_stl
│  ├─ models_info.json
│  ├─ obj_OBJ_ID.stl
########

CAMERA = {photoneo, basler_hr[1-5], basler_lr[1-3], flir_polar[1-4]}
######## TEST ZIP
├─ test
│  ├─ BOP SCENE_ID
│  │  ├─ scene_camera_[CAMERA].json
		- camera info for each IMG_ID
│  │  ├─ scene_gt.json
		- List[6D pose and OBJ_ID in GT_ID order] for each IMG_ID
│  │  ├─ scene_gt_info.json
		- List[bounding boxes in GT_ID order] for each IMG_ID
│  │  ├─ mask
│  │  │  ├─ IMGID_GTID.png
│  │  ├─ mask_visib
│  │  │  ├─ IMGID_GTID.png
│  │  ├─ depth_[CAMERA]
│  │  │  ├─ IMGID.png
│  │  ├─ rgb_[CAMERA] #if multiple resolutions, combined as HDR image
│  │  │  ├─ IMGID.png
######## 
```

## To Install

Suggested to run on a Linux Machine with Nvidia GPU. The ipd toolkit uses `pyrender` for offscreen rendering which recommends EGL as the renderer. EGL comes with Nvidia Drivers. See pyrender docs [https://pyrender.readthedocs.io/en/latest/install/index.html#osmesa].

### Option 1: Install Dev Deps via PDM
1. Install pdm 
2. Clone `ipd` repo 
3. Sync `bop_toolkit` submodule: `git submodule update --init --recursive`
4. `pdm install`

Note: `bop_toolkit` should be an editable install!

### Option 2: Manual Install via pip

#### IPD Toolkit

In [5]:
# !python3 -m pip install -e .

#### BOP Toolkit

In [ ]:
# !git clone git@github.com:thodan/bop_toolkit.git
# !python3 -m pip install -r bop_toolkit/requirements.txt -e bop_toolkit/

#### Other

In [ ]:
# !python3 -m pip install open3d, pymeshlab

# Begin Conversion

## 0. Imports & Setup

In [4]:
import os, shutil
import numpy as np
import open3d as o3d
import pymeshlab
import cv2
import json
from collections import defaultdict

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [5]:
from intrinsic_ipd import IPDReader
import intrinsic_ipd.constants

In [6]:
from bop_toolkit.bop_toolkit_lib import misc, inout

In [7]:
import logging
logging.basicConfig(
    level=logging.INFO,  # Set the logging level (INFO, DEBUG, WARNING, etc.)
    format="%(asctime)s - %(name)s - %(levelname)s - %(filename)s - %(funcName)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

In [8]:
def json_load_if_exists(path_to_json, default):
    if os.path.exists(path_to_json):
        with open(path_to_json, 'r') as fp:
            return json.load(fp, object_hook = lambda d: {int(k) 
                         if k.lstrip('-').isdigit() else k: v for k, v in d.items()})
    else:
        return default

### Read Dataset & Setup Destinations

In [ ]:
# CHOOSE CAMERA AND DATASET

camera = intrinsic_ipd.constants.IPDCamera.PHOTONEO
# camera = intrinsic_ipd.constants.IPDCamera.BASLER_LR1


datasets = intrinsic_ipd.constants.DATASET_IDS
bop_scene = 0 #TODO: ITERATE OVER datasets.keys()
dataset = datasets[bop_scene]

In [ ]:
# DOWNLOAD DATA & READ
reader = IPDReader("./datasets", dataset, camera, lighting=intrinsic_ipd.constants.IPDLightCondition.ALL, download=True) 

Photoneo-dataset_basket_0.zip: 509MB [00:02, 178MB/s]                               


Extracting ./datasets/Photoneo-dataset_basket_0.zip...


100%|██████████| 813/813 [00:03<00:00, 233.69it/s]


Extracted ./datasets/Photoneo-dataset_basket_0.zip to ./datasets


pegboard_basket.stl: 0.00B [00:00, ?B/s]
ERROR:root:url: https://storage.googleapis.com/akasha-public/industrial_plenoptic_dataset/cad_models/pegboard_basket.stl failed to download with error: HTTP Error 404: Not Found
  0%|          | 0/630 [00:00<?, ?it/s]WARNING:root:You should set the PYOPENGL_PLATFORM environment variable before importing pyrender or any other OpenGL library. 
	Setting PYOPENGL_PLATFORM=`egl`. 
	See https://pyrender.readthedocs.io/en/latest/examples/offscreen.html 
[ WARN:0@15.276] global matrix_expressions.cpp:1333 assign OpenCV/MatExpr: processing of multi-channel arrays might be changed in the future: https://github.com/opencv/opencv/issues/16739
100%|██████████| 630/630 [01:38<00:00,  6.37it/s]


In [ ]:
# Make ipd dest
bop_dest = "./datasets_bop"
ipd_dest = os.path.join(bop_dest, "ipd")
os.makedirs(ipd_dest, exist_ok=True)

# Make test dest
test_dest = os.path.join(ipd_dest, "test")
os.makedirs(test_dest, exist_ok=True)

In [12]:
# Make bop_scene map
bop_scene_map_file = os.path.join(ipd_dest, "ipd_dataset_to_bop_scene.json")
bop_scene_map = json_load_if_exists(bop_scene_map_file, {did : i for i, did in enumerate(intrinsic_ipd.constants.DATASET_IDS)})
inout.save_json(bop_scene_map_file, bop_scene_map)

In [13]:
# Make dataset (bop_scene) dest
scene_dest = os.path.join(test_dest, f"{bop_scene:06}")
os.makedirs(scene_dest, exist_ok=True)

## 1. Convert Models (ONLY NEED TO DO THIS ONE TIME FOR THE FIRST DATASET)

In [14]:
# Make models dest
models_stl_dest = os.path.join(ipd_dest, "models_stl")
models_dest = os.path.join(ipd_dest, "models")
models_eval_dest = os.path.join(ipd_dest, "models_eval")
os.makedirs(models_stl_dest, exist_ok=True)
os.makedirs(models_dest, exist_ok=True)
os.makedirs(models_eval_dest, exist_ok=True)

In [ ]:
# CONVERT STL to PLY USING PYMESHLAB
def pymeshlab_stl_to_ply(in_stl_path, out_ply_path):
    ms = pymeshlab.MeshSet()
    ms.load_new_mesh(in_stl_path)
    ms.save_current_mesh(out_ply_path,
                         binary = False,
                         save_vertex_normal = True
                         )
    
# Alternative: CONVERT STL to PLY USING OPEN3D
def o3d_stl_to_ply(in_stl_path, out_ply_path, sample=False, num_points = 10000):
    mesh = o3d.io.read_triangle_mesh(in_stl_path)
    if sample:
        cloud = mesh.sample_points_uniformly(num_points, use_triangle_normal=True)
    else:
        mesh.compute_vertex_normals()
        mesh.paint_uniform_color((1, 0.75, 0))
        mesh.compute_vertex_normals()
        cloud = o3d.geometry.PointCloud()
        cloud.points = mesh.vertices
        cloud.normals = mesh.vertex_normals 
    o3d.io.write_point_cloud(out_ply_path, cloud, write_ascii=True)


In [ ]:
# REMESH PLY FOR EVAL
# TODO: FIX THIS FUNCTION
def remesh_for_eval(in_mesh_path, out_mesh_path, cell=0.25):
    """
    'Uniformly' resamples and decimates 3D object models for evaluation. 
    See bop_toolkit/scripts/remesh_models_for_eval.py
    !!!! DOES NOT WORK !!!! some error with pymeshlab or arguments? 

    Parameters
    ----------
    in_mesh_path : string
        where the input mesh is located
    out_mesh_path : string
        where the output mesh will be saved
    cell : float, optional
        cell size of resampling, by default 0.25
    """
    # Refactored from ipd/bop_toolkit/scripts/meshlab_scripts/remesh_for_eval_cell=0.25.mlx
    ms = pymeshlab.MeshSet()
    ms.load_new_mesh(in_mesh_path)
    ms.meshing_remove_unreferenced_vertices()
    ms.meshing_remove_duplicate_vertices()
    ms.meshing_remove_duplicate_faces()
    ms.generate_resampled_uniform_mesh(
        cellsize=pymeshlab.PercentageValue(0.50), #Need to ask BOP people about this
        offset=pymeshlab.PercentageValue(0), #Need to ask BOP people about this
        mergeclosevert=True,
        discretize=False,
        multisample=True,
        absdist=False,
    )
    ms.meshing_decimation_quadric_edge_collapse(
        targetfacenum=0, #Need to ask BOP people about this
        targetperc=0.025, #Need to ask BOP people about this
        qualitythr=0.5, #Need to ask BOP people about this
        preserveboundary=True,
        boundaryweight=1,
        preservenormal=True,
        preservetopology=False,
        optimalplacement=True,
        planarquadric=True,
        qualityweight=False,
        autoclean=True,
        selected=False,
    )
        
    ms.save_current_mesh(out_mesh_path,
                        binary = False,
                        save_vertex_normal = True
                        )


In [ ]:
# CONVERT SYMMETRY FROM IPD TO BOP
# TODO: CHECK RESULTS OF THIS FUNCTION (see very bottom)
def convert_symmetries(symmetry_params):
    model_info = {
        "symmetries_discrete": [],
        "symmetries_continuous": []
    }
    if symmetry_params is None:
        return model_info

    # Convert proper_symms into a list of flattened 4x4 matrices
    if symmetry_params.get("proper_symms", None) is not None:
        for matrix in symmetry_params["proper_symms"]:
            # Convert 3x3 to 4x4 by adding identity row/column
            matrix_4x4 = np.eye(4)
            matrix_4x4[:3, :3] = matrix
            # Flatten and append to symmetries_discrete
            if not np.allclose(matrix_4x4, np.eye(4)):  # Exclude identity
                model_info["symmetries_discrete"].append(matrix_4x4.flatten().tolist())

    # Convert continuous_symm_axis to a list of continuous symmetries dictionaries
    if symmetry_params["continuous_symm_axis"] != -1:
        axis_index = symmetry_params["continuous_symm_axis"]
        axis_vectors = [
            [1, 0, 0],  # x-axis
            [0, 1, 0],  # y-axis
            [0, 0, 1]   # z-axis
        ]
        model_info["symmetries_continuous"].append({
            "axis": axis_vectors[axis_index],
            "offset": [0, 0, 0]
        })

    return model_info

In [ ]:
##################  BEGIN CONVERSION OF MODELS ##################
TEST = False # If true, only convert first object
REMESH = False
#################################################################


models_info = {}
sample = False
parts = intrinsic_ipd.constants.PART_NAMES
if TEST:
    parts = intrinsic_ipd.constants.PART_NAMES[0:1]

part2obid = {}
for obj_id, part in enumerate(parts):
    # obj_id += 1 # index starts with 1
    part2obid[part] = obj_id

    ######## SAVE MODEL
    # copy stl model to models_stl
    model_stl_path = os.path.join(reader.root, 'models', f'{part}.stl')
    if not os.path.exists(model_stl_path): 
        print(f"NO STL FILE FOUND FOR: {part} at {model_stl_path}")
        continue
    dst = os.path.join(models_stl_dest, f'obj_{obj_id:06}.stl')
    shutil.copyfile(model_stl_path, dst)

    # create ply model in models
    model_ply_path = os.path.join(models_dest, f'obj_{obj_id:06}.ply')
    pymeshlab_stl_to_ply(model_stl_path, model_ply_path)
    
    # create models_eval via remeshing
    if REMESH:
        remesh_for_eval(model_ply_path, os.path.join(models_eval_dest, f'obj_{obj_id:06}.ply'))

    ######## SAVE MODEL INFO: see bop_toolkit/scripts/calc_model_info.py
    model = inout.load_ply(model_ply_path)
    ref_pt = list(map(float, model["pts"].min(axis=0).flatten()))
    size = list(map(float, (model["pts"].max(axis=0) - ref_pt).flatten()))
    diameter = misc.calc_pts_diameter(model["pts"])

    model_info = {
        "min_x": ref_pt[0],
        "min_y": ref_pt[1],
        "min_z": ref_pt[2],
        "size_x": size[0],
        "size_y": size[1],
        "size_z": size[2],
        "diameter": diameter,
    }

    # Process symmetries
    # see: https://github.com/thodan/bop_toolkit/blob/97badc48dae87d03fa86c0f4ccce94ffdaaae4c5/bop_toolkit_lib/misc.py#L47
    # TODO: check if this is correct by running bop_toolkit/scripts/vis_object_symmetries.py, seems to work, but results look shoddy
    symm = reader._get_symm_params(part)
    converted_symm = convert_symmetries(symm)
    # list of continuous symmetries arrays
    model_info['symmetries_discrete'] = converted_symm['symmetries_discrete']
    # list of continuous symmetries dictionaries
    model_info['symmetries_continuous'] = converted_symm['symmetries_continuous']


    models_info[obj_id] = model_info

inout.save_json(os.path.join(models_dest, 'models_info.json'), models_info)
inout.save_json(os.path.join(models_stl_dest, 'models_info.json'), models_info)
# inout.save_json(os.path.join(models_eval_dest, 'models_info.json'), models_info) # TODO: create eval ply

inout.save_json(os.path.join(ipd_dest, 'ipd_part_to_bop_obj_id.json'), part2obid)

NO STL FILE FOUND FOR: corner_bracket5 at ./datasets/models/corner_bracket5.stl
NO STL FILE FOUND FOR: pegboard_basket at ./datasets/models/pegboard_basket.stl


## 2. Convert IPD object into BOP ground truth instances

In [21]:
def get_bop_gt_id_map(reader, scene_dest):
    map_json = os.path.join(scene_dest, 'ipd_obj_to_bop_gt_id.json')

    bop_gt_id_map = json_load_if_exists(map_json, defaultdict(dict))

    for i, obj in enumerate(reader.objects):
          bop_gt_id_map[obj[0]][obj[1]] = i
    with open(map_json, 'w') as fp:
            json.dump(bop_gt_id_map, fp, sort_keys=True, indent=4)
    return bop_gt_id_map

get_bop_gt_id_map(reader, scene_dest)

{'gear1': {0: 0, 1: 1, 2: 2, 3: 3},
 'pegboard_basket': {0: 4},
 'u_bolt': {0: 5, 1: 6, 2: 7}}

## 3. Move/Convert Images

In [22]:
# Make image dests
if camera == intrinsic_ipd.constants.IPDCamera.PHOTONEO:
    depth_dest = os.path.join(scene_dest, f'depth_{camera.name.lower()}')
    os.makedirs(depth_dest, exist_ok=True)
mask_dest = os.path.join(scene_dest, f'mask_ipd_{camera.name.lower()}')
rgb_dest = os.path.join(scene_dest, f'rgb_{camera.name.lower()}')


os.makedirs(mask_dest, exist_ok=True)
os.makedirs(rgb_dest, exist_ok=True)

### Move RBGD images

In [23]:
def merge_exposures(img_paths):
    img_list = [cv2.imread(path) for path in img_paths]
    exposure_times = np.array([1, 30, 80, 200], dtype=np.float32)
    merge_debevec = cv2.createMergeDebevec()
    hdr_debevec = merge_debevec.process(img_list, times=exposure_times.copy())
    return hdr_debevec

For photoneo, will move & rename rgb and depth files.

For other cameras, will merge into an hdr photo and save

In [24]:
for bop_image_id in reader.scenes.keys():
    # move or merge rgb & depth photos
    if reader.camera == intrinsic_ipd.IPDCamera.PHOTONEO:
        from_path = reader._get_img_file(bop_image_id, intrinsic_ipd.IPDImage.PHOTONEO_DEPTH)
        to_path = os.path.join(depth_dest, f'{bop_image_id:06}.png')
        shutil.copy(from_path, to_path)

        from_path = reader._get_img_file(bop_image_id, intrinsic_ipd.IPDImage.PHOTONEO_HDR)
        to_path = os.path.join(rgb_dest, f'{bop_image_id:06}.png')
        shutil.copy(from_path, to_path)
    else:
        img_paths = [reader._get_img_file(bop_image_id, image_type) for image_type in reader.camera.images]
        hdr_photo = merge_exposures(img_paths)
        to_path = os.path.join(rgb_dest, f'{bop_image_id:06}.png')
        cv2.imwrite(to_path, hdr_photo)
    


### Move masks based on ground truth id

Move and rename masks based on ground truth id.

In [25]:
# move and rename mask photos
bop_gt_id_map = None
for bop_image_id in reader.scenes.keys():
    for object in reader.objects:
        part, instance = object
        bop_gt_id_map = get_bop_gt_id_map(reader, scene_dest)
        bop_gt_id = bop_gt_id_map[part][instance]
        try:
            _, ipd_mask_path = reader.get_mask(bop_image_id, part, instance, return_path=True)
            bop_mask_path = os.path.join(mask_dest, f'{bop_image_id:06}_{bop_gt_id:06}.png')
            shutil.copy(ipd_mask_path, bop_mask_path)
        except:
            print(f"NO MASK FILE FOUND FOR: {part} at {ipd_mask_path}")
            continue

NO MASK FILE FOUND FOR: pegboard_basket at ./datasets/dataset_basket_0/test/000000/000/mask/gear1/3.png
NO MASK FILE FOUND FOR: pegboard_basket at ./datasets/dataset_basket_0/test/000001/000/mask/gear1/3.png
NO MASK FILE FOUND FOR: pegboard_basket at ./datasets/dataset_basket_0/test/000002/000/mask/gear1/3.png
NO MASK FILE FOUND FOR: pegboard_basket at ./datasets/dataset_basket_0/test/000003/000/mask/gear1/3.png
NO MASK FILE FOUND FOR: pegboard_basket at ./datasets/dataset_basket_0/test/000004/000/mask/gear1/3.png
NO MASK FILE FOUND FOR: pegboard_basket at ./datasets/dataset_basket_0/test/000005/000/mask/gear1/3.png
NO MASK FILE FOUND FOR: pegboard_basket at ./datasets/dataset_basket_0/test/000006/000/mask/gear1/3.png
NO MASK FILE FOUND FOR: pegboard_basket at ./datasets/dataset_basket_0/test/000007/000/mask/gear1/3.png
NO MASK FILE FOUND FOR: pegboard_basket at ./datasets/dataset_basket_0/test/000008/000/mask/gear1/3.png
NO MASK FILE FOUND FOR: pegboard_basket at ./datasets/dataset_ba

## 4. Process Labels

### scene_camera.json
cam_K, cam_R_w2c, cam_t_w2c, depth_scale, elev, mode 
for each bop image (ipd scene)

In [26]:
scene_camera_path = os.path.join(scene_dest, f'scene_camera_{reader.camera.name.lower()}.json')
camera_path = os.path.join(ipd_dest, f"camera_{reader.camera.name.lower()}.json")

In [27]:
w2c = np.linalg.inv(reader.cam_c2w)
camera_info = {
    'cam_K': reader.cam_K.flatten().tolist(),
    'cam_R_w2c': w2c[:3,:3].flatten().tolist(),
    'cam_t_w2c': w2c[:3,3].flatten().tolist(),
}

if reader.camera is intrinsic_ipd.constants.IPDCamera.PHOTONEO:
    camera_info['depth_scale']= 1.0

image = reader.get_img(list(reader.scenes.keys())[0])
height, width = image.shape[:2]

scene_camera = {bop_image_id : camera_info  for bop_image_id in reader.scenes.keys()}
inout.save_json(scene_camera_path, scene_camera)
inout.save_json(camera_path, {
  "cx": reader.cam_K[0, 2],
  "cy": reader.cam_K[1, 2],
  "depth_scale": 1.0,
  "fx": reader.cam_K[0, 0],
  "fy": reader.cam_K[1, 1],
  "height": height,
  "width": width
})

### scene_gt.json
map IMG_ID to List[6D pose and OBJ_ID in GT_ID order]

In [28]:
scene_gt_path = os.path.join(scene_dest, f'scene_gt_{reader.camera.name.lower()}.json')

In [29]:
def get_gt_info(reader, ipd_scene_id, ipd_dest, scene_dest):
    with open(os.path.join(ipd_dest, "ipd_part_to_bop_obj_id.json"), 'r') as fp:
        bop_obj_id_map = json.load(fp)
    ipd_objects = reader.objects
    o2c = reader.o2c.sel(scene=ipd_scene_id)
    gt_info = {}
    bop_gt_id_map = get_bop_gt_id_map(reader, scene_dest)
    for part, instance in ipd_objects:
        gt_id = bop_gt_id_map[part][instance]
        gt_o2c = o2c.sel(part=part, instance=instance).data
        obj_id = bop_obj_id_map[part]
        gt_info[gt_id] = {
            'obj_id': obj_id, 
            'cam_R_m2c': gt_o2c[:3,:3].flatten().tolist(),
            'cam_t_m2c': gt_o2c[:3, 3].flatten().tolist(),
            'gt_id': gt_id,
            'ipd_object': (part, instance)
        }
    gt_keys = [int(k) for k in gt_info.keys()]
    max_gt_id = max(gt_keys)
    return [gt_info.get(gt_id, {}) for gt_id in range(max_gt_id)]


In [30]:
scene_gt = {bop_image_id: get_gt_info(reader, bop_image_id, ipd_dest, scene_dest) for bop_image_id in reader.scenes.keys()}
inout.save_json(scene_gt_path, scene_gt)

#### test_targets.json

In [31]:
import itertools, operator

bop_obj_id_map = json_load_if_exists(os.path.join(ipd_dest, 'ipd_part_to_bop_obj_id.json'), {})


conditions = [intrinsic_ipd.constants.IPDLightCondition.ALL,
              intrinsic_ipd.constants.IPDLightCondition.DAY, 
              intrinsic_ipd.constants.IPDLightCondition.ROOM,
              intrinsic_ipd.constants.IPDLightCondition.SPOT]

bop_scene = json_load_if_exists(os.path.join(ipd_dest, 'ipd_dataset_to_bop_scene.json'), {})[reader.dataset_id]
for condition in conditions:
    if condition is intrinsic_ipd.constants.IPDLightCondition.ALL:
        target_file = os.path.join(ipd_dest, f'test_targets_bop19.json')
        targets_24_file = os.path.join(ipd_dest, f'test_targets_bop24.json')
    else:
        target_file = os.path.join(ipd_dest, f'test_targets_bop19_{condition.name.lower()}.json')
        targets_24_file = os.path.join(ipd_dest, f'test_targets_bop24_{condition.name.lower()}.json')
    targets = json_load_if_exists(target_file, [])    
    for part, objects in itertools.groupby(reader.objects, operator.itemgetter(0)):
        obj_id = bop_obj_id_map[part]
        inst_count = len(list(objects))
        for bop_image_id in condition.scenes:
            target = {
                "im_id": bop_image_id,
                "obj_id": obj_id,
                "inst_count": inst_count,
                "scene_id": bop_scene,
            }
            if target not in targets:
                targets.append(target)
    
    inout.save_json(target_file, targets)
    
    
    targets_24 = json_load_if_exists(targets_24_file, [])
    for bop_image_id in condition.scenes:
        target_24 = {
                "im_id": bop_image_id,
                "scene_id": bop_scene,
            }
        if target_24 not in targets_24:
            targets_24.append(target_24)
    inout.save_json(targets_24_file, targets_24)
    

## 5. Repeat above steps for all dataset ids and camera (except converting models)

## 6.Run BOP scripts to generate rest of dataset info 

To run the bop_toolkit scripts, need to make some edits:

Changelog to `bop_toolkit` as reflected in @carynbear's fork:
- in `bop_toolkit/bop_toolkit_lib/dataset_params.py`
    - CHANGED: added `ipd` params throughout
    - TODO: indicate which objects (parts) have symmetry.
    - TODO: get sizes of images for other cameras (photoneo done)
    - TODO: calculate depth_range, azimuth_range, elev_range 
- in `bop_toolkit/bop_toolkit_lib/config.py`
    - CHANGED: `output_path`
- in `bop_toolkit/scripts/calc_gt_info.py`
    - CHANGED: run with `ipd` and `vis`
    - CHANGED: try catch to skip missing cad models
- in `bop_toolkit/scripts/calc_gt_masks.py`
    - CHANGED: run with `ipd`
    - CHANGED: try catch to skip missing cad models

### Visualize ground truth poses

ToDo:
- upload the missing model (pegboard basket)

To Run with X11 forwarding:
- install XQuartz server on local
- enable X11 forwarding over ssh
    ForwardX11 yes
    ForwardAgent yes
    ForwardX11Trusted yes
- Touch .Xauthority file on remote
- Set Display Env Variable if not already set.

In [45]:
!DISPLAY=127.0.0.1:10 BOP_PATH=./datasets_bop python3 ./bop_toolkit/scripts/vis_gt_poses.py -v

                mandatory if you are running evaluation on HOT3d.
                Refer to the README.md for installation instructions.
                
                mandatory if you are running evaluation on HOT3d.
                Refer to the README.md for installation instructions.
                
{'name': 'ipd', 'split': 'test', 'split_type': None, 'base_path': './datasets_bop/ipd', 'depth_range': None, 'azimuth_range': None, 'elev_range': None, 'im_modalities': ['rgb_photoneo', 'depth_photoneo'], 'eval_modality': <function get_split_params.<locals>.ipd_eval_modality at 0x7fadf629f040>, 'test_scene_ids': [0], 'scene_ids': [0], 'photoneo_im_size': (2064, 1544), 'im_size': (2064, 1544), 'split_path': './datasets_bop/ipd/test', 'supported_error_types': ['ad', 'add', 'adi', 'mssd', 'mspd'], 'rgb_photoneo_tpath': './datasets_bop/ipd/test/{scene_id:06d}/rgb_photoneo/{im_id:06d}.png', 'scene_camera_rgb_photoneo_tpath': './datasets_bop/ipd/test/{scene_id:06d}/scene_camera_photoneo.json

### Calculate scene_gt_info.json
map IMG_ID to List[bounding boxes in GT_ID order]

- See `https://github.com/thodan/bop_toolkit/issues/137`

In [46]:
!DISPLAY=127.0.0.1:10 BOP_PATH=./datasets_bop python3 ./bop_toolkit/scripts/calc_gt_info.py

{'name': 'ipd', 'split': 'test', 'split_type': None, 'base_path': './datasets_bop/ipd', 'depth_range': None, 'azimuth_range': None, 'elev_range': None, 'im_modalities': ['rgb_photoneo', 'depth_photoneo'], 'eval_modality': <function get_split_params.<locals>.ipd_eval_modality at 0x7ff55ee43040>, 'test_scene_ids': [0], 'scene_ids': [0], 'photoneo_im_size': (2064, 1544), 'im_size': (2064, 1544), 'split_path': './datasets_bop/ipd/test', 'supported_error_types': ['ad', 'add', 'adi', 'mssd', 'mspd'], 'rgb_photoneo_tpath': './datasets_bop/ipd/test/{scene_id:06d}/rgb_photoneo/{im_id:06d}.png', 'scene_camera_rgb_photoneo_tpath': './datasets_bop/ipd/test/{scene_id:06d}/scene_camera_photoneo.json', 'scene_gt_rgb_photoneo_tpath': './datasets_bop/ipd/test/{scene_id:06d}/scene_gt_photoneo.json', 'scene_gt_info_rgb_photoneo_tpath': './datasets_bop/ipd/test/{scene_id:06d}/scene_gt_info_photoneo.json', 'scene_gt_coco_rgb_photoneo_tpath': './datasets_bop/ipd/test/{scene_id:06d}/scene_gt_coco_photoneo.js

### Generate masks from gt labels
- See `https://github.com/thodan/bop_toolkit/issues/137`

In [33]:
!BOP_PATH=./datasets_bop python3 ./bop_toolkit/scripts/calc_gt_masks.py

{'name': 'ipd', 'split': 'test', 'split_type': None, 'base_path': './datasets_bop/ipd', 'depth_range': None, 'azimuth_range': None, 'elev_range': None, 'im_modalities': ['rgb_photoneo', 'depth_photoneo'], 'eval_modality': <function get_split_params.<locals>.ipd_eval_modality at 0x7fd3d2ec4040>, 'test_scene_ids': [0], 'scene_ids': [0], 'photoneo_im_size': (2064, 1544), 'im_size': (2064, 1544), 'split_path': './datasets_bop/ipd/test', 'supported_error_types': ['ad', 'add', 'adi', 'mssd', 'mspd'], 'rgb_photoneo_tpath': './datasets_bop/ipd/test/{scene_id:06d}/rgb_photoneo/{im_id:06d}.png', 'scene_camera_rgb_photoneo_tpath': './datasets_bop/ipd/test/{scene_id:06d}/scene_camera_photoneo.json', 'scene_gt_rgb_photoneo_tpath': './datasets_bop/ipd/test/{scene_id:06d}/scene_gt_photoneo.json', 'scene_gt_info_rgb_photoneo_tpath': './datasets_bop/ipd/test/{scene_id:06d}/scene_gt_info_photoneo.json', 'scene_gt_coco_rgb_photoneo_tpath': './datasets_bop/ipd/test/{scene_id:06d}/scene_gt_coco_photoneo.js

### Generate models_evel with pymeshlab
- FileNotFoundError: [Errno 2] No such file or directory: '/path/to/meshlabserver.exe'
- See `https://github.com/thodan/bop_toolkit/issues/137`

In [35]:
!BOP_PATH=./datasets_bop python3 ./bop_toolkit/scripts/remesh_models_for_eval.py

1/9|13:57:40: 


Processing model of object 1...

1/9|13:57:40: /path/to/meshlabserver.exe -s /home/ngan/ipd/bop_toolkit/scripts/meshlab_scripts/remesh_for_eval_cell=0.25.mlx -i ./datasets_bop/lm/models/obj_000001.ply -o ./datasets_bop/lm/models_eval/obj_000001.ply
Traceback (most recent call last):
  File "/home/ngan/ipd/./bop_toolkit/scripts/remesh_models_for_eval.py", line 63, in <module>
    misc.run_meshlab_script(
  File "/home/ngan/ipd/bop_toolkit/bop_toolkit_lib/misc.py", line 422, in run_meshlab_script
    if subprocess.call(meshlabserver_cmd) != 0:
  File "/usr/lib/python3.9/subprocess.py", line 349, in call
    with Popen(*popenargs, **kwargs) as p:
  File "/usr/lib/python3.9/subprocess.py", line 951, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "/usr/lib/python3.9/subprocess.py", line 1823, in _execute_child
    raise child_exception_type(errno_num, err_msg, err_filename)
FileNotFoundError: [Errno 2] No such file or directory: '/path/t

### Generate training images
- See `https://github.com/thodan/bop_toolkit/issues/137`

In [ ]:
!BOP_PATH=./datasets_bop python3 ./bop_toolkit/scripts/render_train_imgs.py

Traceback (most recent call last):
  File "/home/ngan/ipd/./bop_toolkit/scripts/render_train_imgs.py", line 89, in <module>
    dp_camera = dataset_params.get_camera_params(datasets_path, dataset, cam_type)
  File "/home/ngan/ipd/bop_toolkit/bop_toolkit_lib/dataset_params.py", line 68, in get_camera_params
    p.update(inout.load_cam_params(cam_params_path))
  File "/home/ngan/ipd/bop_toolkit/bop_toolkit_lib/inout.py", line 142, in load_cam_params
    c = load_json(path)
  File "/home/ngan/ipd/bop_toolkit/bop_toolkit_lib/inout.py", line 84, in load_json
    f = open(path, "r")
FileNotFoundError: [Errno 2] No such file or directory: '/path/to/bop/datasets/tyol/camera.json'


### Inspect Symmetry Annotations Visually

In [ ]:
!BOP_PATH=./datasets_bop python3 ./bop_toolkit/scripts/vis_object_symmetries.py